# WeatherAPI

This notebook demonstrates how to collect weather data without storing the API key in Git. The key must stay only in the local `.env` file.

In [ ]:
import os
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(project_root / ".env")

API_KEY = os.getenv("WEATHER_API_KEY")
LOCATION = os.getenv("WEATHER_API_LOCATION", "41.8345,-7.7889")
DAYS_TO_FETCH = int(os.getenv("WEATHER_API_DAYS", "44"))
END_DATE = os.getenv("WEATHER_API_END_DATE")

if not API_KEY:
    raise RuntimeError("Set WEATHER_API_KEY in the local .env file.")

In [ ]:
end = pd.to_datetime(END_DATE).to_pydatetime() if END_DATE else datetime.today()
dates = [(end - timedelta(days=i)).strftime("%Y-%m-%d") for i in range(1, DAYS_TO_FETCH + 1)]

records = []
endpoint = "https://api.weatherapi.com/v1/history.json"

for date in dates:
    params = {"key": API_KEY, "q": LOCATION, "dt": date}
    response = requests.get(endpoint, params=params, timeout=10)
    if response.status_code == 200:
        data = response.json()
        forecast_day = data["forecast"]["forecastday"][0]
        day = forecast_day["day"]
        records.append(
            {
                "date": date,
                "temp_c": day["avgtemp_c"],
                "wind_kph": day["maxwind_kph"],
                "wind_dir_deg": forecast_day["hour"][12]["wind_degree"],
            }
        )
    else:
        print(f"Error fetching data for {date}: HTTP {response.status_code}")

df = pd.DataFrame(records)
df

In [ ]:
endpoint = "https://api.weatherapi.com/v1/current.json"
params = {"key": API_KEY, "q": LOCATION, "aqi": "no"}
response = requests.get(endpoint, params=params, timeout=10)
response.raise_for_status()
data = response.json()

temperature = data["current"]["temp_c"]
wind_kph = data["current"]["wind_kph"]
wind_degrees = data["current"]["wind_degree"]

print(f"Location: {data['location']['name']}")
print(f"Temperature: {temperature} °C")
print(f"Wind speed: {wind_kph} km/h")
print(f"Wind direction: {wind_degrees}°")